In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
import json

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=groq_api_key
)

In [2]:
def get_video_id(url):
    if "v=" in url:
        video_id = url.split("v=")[1]
        video_id = video_id.split("&")[0]
        return video_id
    elif "youtu.be/" in url:
        video_id = url.split("youtu.be/")[1]
        video_id = video_id.split("?")[0]
        return video_id
    else:
        return None

def get_transcript(url):
    video_id = get_video_id(url)
    if not video_id:
        print("Invalid Youtube URL")
        return None
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id)
        return transcript
    except Exception as e:
        print("Error: Could not fetch transcript. The video may not have captions or may not exist.")
        return None

def chunk_transcript(transcript, chunk_duration=120):
    chunks = []
    current_text = ""
    chunk_start = transcript[0].start

    for segment in transcript:
        if segment.start - chunk_start >= chunk_duration:
            chunks.append({
                "text": current_text.strip(),
                "start": chunk_start,
                "end": segment.start
            })
            current_text = segment.text + " "
            chunk_start = segment.start
        else:
            current_text += segment.text + " "

    # Don't forget the last chunk that's still being built
    if current_text:
        chunks.append({
            "text": current_text.strip(),
            "start": chunk_start,
            "end": transcript[-1].start
        })

    return chunks


class TranscriptionAgent:
    def __init__(self):
        self.name = "Transcription Agent"
    
    def run(self, url):
        transcript = get_transcript(url)
        if not transcript:
            return None
        chunks = chunk_transcript(transcript)
        return chunks

In [3]:
agent1 = TranscriptionAgent()
chunks = agent1.run("https://www.youtube.com/watch?v=aircAruvnKk")
print(f"Agent: {agent1.name}")
print(f"Chunks generated: {len(chunks)}")

Agent: Transcription Agent
Chunks generated: 10


In [29]:
class FilterAgent:
    def __init__(self, llm):
        self.name = "Filter Agent"
        self.llm = llm
    
    def run(self, chunks):
        educational_chunks = []
        for chunk in chunks:
            try:
                prompt = f"""Look at this transcript section and decide if it contains actual educational content or if it is filler content.

Filler content includes: introductions, outros, subscribe reminders, agenda overview, quiz segments, promotions, greetings, channel promotions, "like and share" requests, or any non-teaching content.

IMPORTANT: If the section contains ANY educational explanation, concepts, or teaching, even if mixed with casual talk, mark it as "educational".
Only mark as "skip" if the ENTIRE section is filler with NO educational value.

Transcript:
{chunk['text']}

Reply with ONLY one word: "educational" or "skip"
"""
                response = self.llm.invoke(prompt)
                content = response.content.strip().lower()
                if content == "educational":
                    educational_chunks.append(chunk)
                else:
                    print(f"Skipping filler chunk at {chunk['start']:.0f}s")
            except Exception as e:
                print(f"Error filtering chunk: {e}")
                educational_chunks.append(chunk)
        
        print(f"Kept {len(educational_chunks)} out of {len(chunks)} chunks")
        return educational_chunks
        # For each chunk:
        #   Ask LLM if it's educational or filler
        #   If educational, keep it
        #   If filler, skip it
        # Return only educational chunks

In [30]:
agent_filter = FilterAgent(llm)
filtered_chunks = agent_filter.run(chunks)

Kept 10 out of 10 chunks


In [31]:
agent1 = TranscriptionAgent()
krish_chunks = agent1.run("https://www.youtube.com/watch?v=_gGRBWaRpAQ&list=PLZoTAELRMXVNNrHSKv36Lr3_156yCo6Nn&index=5")
print(f"Total chunks: {len(krish_chunks)}")

agent_filter = FilterAgent(llm)
filtered_chunks = agent_filter.run(krish_chunks)

Total chunks: 37
Skipping filler chunk at 14s
Skipping filler chunk at 134s
Skipping filler chunk at 254s
Skipping filler chunk at 3295s
Skipping filler chunk at 3415s
Skipping filler chunk at 3537s
Skipping filler chunk at 3780s
Skipping filler chunk at 4146s
Skipping filler chunk at 4267s
Skipping filler chunk at 4391s
Kept 27 out of 37 chunks


In [18]:
class ConceptExtractionAgent:
    def __init__(self, llm):
        self.name = "Concept Extraction Agent"
        self.llm = llm
    
    def run(self, chunks):
        all_concepts = []
        
        for chunk in chunks:
            try:
                start_min = int(chunk["start"] // 60)
                start_sec = int(chunk["start"] % 60)
                end_min = int(chunk["end"] // 60)
                end_sec = int(chunk["end"] % 60)
                timestamp = f"{start_min}:{start_sec:02d} - {end_min}:{end_sec:02d}"
                
                prompt = f"""Extract the 2-3 most important concepts or topics from this transcript section.

Transcript:
{chunk['text']}

Return ONLY JSON in this format:
{{"concepts": ["concept 1", "concept 2", "concept 3"]}}
"""
                response = self.llm.invoke(prompt)
                content = response.content.strip()
                if content.startswith("```json"):
                    content = content[7:]
                if content.startswith("```"):
                    content = content[3:]
                if content.endswith("```"):
                    content = content[:-3]
                content = content.strip()
                
                concepts = json.loads(content)
                concepts["timestamp"] = timestamp
                all_concepts.append(concepts)
            except Exception as e:
                print(f"Skipping chunk: {e}")
                continue
        
        return all_concepts

In [5]:
agent2 = ConceptExtractionAgent(llm)
concepts = agent2.run(chunks)

for c in concepts:
    print(f"\n[{c['timestamp']}]")
    print(f"Concepts: {c['concepts']}")


[0:04 - 2:05]
Concepts: ['Neural Networks', 'Machine Learning', 'Image Recognition']

[2:05 - 4:06]
Concepts: ['Neural Networks', 'Neurons and Activations', 'Network Layers']

[4:06 - 6:07]
Concepts: ['Neural Network Structure', 'Digit Recognition', 'Layered Information Processing']

[6:07 - 8:08]
Concepts: ['Layered Neural Network Structure', 'Feature Detection and Recognition', 'Hierarchical Abstraction in Image Recognition']

[8:08 - 10:10]
Concepts: ['Neural Network Architecture', 'Weighted Sum', 'Feature Extraction']

[10:10 - 12:12]
Concepts: ['Sigmoid Function', 'Neural Network Weights and Biases', 'Activation of Neurons']

[12:12 - 14:14]
Concepts: ['Neural Network Architecture', 'Weights and Biases', 'Network Training and Learning']

[14:14 - 16:15]
Concepts: ['Linear Algebra', 'Neural Network Architecture', 'Matrix Vector Multiplication']

[16:15 - 18:15]
Concepts: ['Neural Network Training', 'Sigmoid Function', 'ReLU (Rectified Linear Unit)']

[18:15 - 18:25]
Concepts: ['Re

In [19]:
class QuestionGenerationAgent:
    def __init__(self, llm):
        self.name = "Question Generation Agent"
        self.llm = llm
    
    def run(self, concepts_list):
        all_questions = []
        
        for item in concepts_list:
            try:
                concepts = ", ".join(item["concepts"])
                timestamp = item["timestamp"]
                
                prompt = f"""Generate 1 multiple choice question that tests understanding of these concepts: {concepts}

The question should have:
- 4 options (A, B, C, D)
- The correct answer
- An explanation of why that answer is correct

Return ONLY JSON in this format:
{{
    "question": "the question text",
    "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
    "correct_answer": "B",
    "explanation": "why this is correct"
}}
"""
                response = self.llm.invoke(prompt)
                content = response.content.strip()
                if content.startswith("```json"):
                    content = content[7:]
                if content.startswith("```"):
                    content = content[3:]
                if content.endswith("```"):
                    content = content[:-3]
                content = content.strip()
                
                question = json.loads(content)
                question["timestamp"] = timestamp
                question["concepts"] = item["concepts"]
                all_questions.append(question)
            except Exception as e:
                print(f"Skipping question: {e}")
                continue
        
        return {"questions": all_questions}

In [7]:
agent3 = QuestionGenerationAgent(llm)
quiz = agent3.run(concepts)

for q in quiz["questions"]:
    print(f"\n[{q['timestamp']}] Concepts: {q['concepts']}")
    print(f"Q: {q['question']}")


[0:04 - 2:05] Concepts: ['Neural Networks', 'Machine Learning', 'Image Recognition']
Q: What type of machine learning model is commonly used for image recognition tasks, such as classifying objects in images?

[2:05 - 4:06] Concepts: ['Neural Networks', 'Neurons and Activations', 'Network Layers']
Q: What is the primary purpose of the activation function in a neural network neuron?

[4:06 - 6:07] Concepts: ['Neural Network Structure', 'Digit Recognition', 'Layered Information Processing']
Q: In a neural network designed for digit recognition, which of the following best describes the role of the hidden layers in the network's structure?

[6:07 - 8:08] Concepts: ['Layered Neural Network Structure', 'Feature Detection and Recognition', 'Hierarchical Abstraction in Image Recognition']
Q: In a layered neural network structure for image recognition, what is the primary role of the early layers in the hierarchy?

[8:08 - 10:10] Concepts: ['Neural Network Architecture', 'Weighted Sum', 'Feat

In [34]:
class EvaluationAgent:
    def __init__(self, llm):
        self.name = "Evaluation Agent"
        self.llm = llm

    def deduplicate_topics(self, weak_topics):
        prompt = f"""Here is a list of topics:
{weak_topics}

Group similar or related topics together and give me a clean, 
deduplicated list. Combine topics that mean the same thing into 
one clear topic name.

Return ONLY JSON in this format:
{{"topics": ["topic 1", "topic 2", "topic 3"]}}
"""
        try:
            response = self.llm.invoke(prompt)
            content = response.content.strip()
            if content.startswith("```json"):
                content = content[7:]
            if content.startswith("```"):
                content = content[3:]
            if content.endswith("```"):
                content = content[:-3]
            content = content.strip()
        
            result = json.loads(content)
            return result["topics"]
        except Exception as e:
            print(f"Deduplication failed: {e}")
            return weak_topics
    
    def run(self, quiz_data):
        score = 0
        weak_topics = []
        
        for i, q in enumerate(quiz_data["questions"]):
            print(f"\nQuestion {i+1}: {q['question']}")
            for option, text in q["options"].items():
                print(f"  {option}) {text}")
            user_answer = input("\nYour answer (A/B/C/D): ").upper()
            if user_answer == q['correct_answer']:
                score = score + 1
                print("Correct!")
            else:
                print("Incorrect!")
                weak_topics.extend(q['concepts'])
                weak_topics = list(set(weak_topics))
            print(f"Correct Answer: {q['correct_answer']}")
            print(f"Explanation: {q['explanation']}")
            print(f"Review in video at: {q['timestamp']}")
            
        print(f"\nFinal Score: {score}/{len(quiz_data['questions'])}")
        if weak_topics:
            weak_topics = self.deduplicate_topics(weak_topics)
        print(f"\nWeak Topics: {weak_topics}")

        return {
            "score": score,
            "total": len(quiz_data["questions"]),
            "weak_topics": weak_topics
        }

In [11]:
# Agent 1: Get transcript
agent1 = TranscriptionAgent()
chunks = agent1.run("https://www.youtube.com/watch?v=aircAruvnKk")
print(f"Agent 1 done: {len(chunks)} chunks")

# Agent 2: Extract concepts
agent2 = ConceptExtractionAgent(llm)
concepts = agent2.run(chunks)
print(f"Agent 2 done: {len(concepts)} concept sets")

# Agent 3: Generate questions
agent3 = QuestionGenerationAgent(llm)
quiz = agent3.run(concepts)
print(f"Agent 3 done: {len(quiz['questions'])} questions")

# Agent 4: Evaluate
agent4 = EvaluationAgent(llm)
results = agent4.run(quiz)

Agent 1 done: 10 chunks
Agent 2 done: 10 concept sets
Agent 3 done: 10 questions

Question 1: What type of machine learning model is commonly used for image recognition tasks, such as classifying objects in images?
  A) Linear Regression
  B) Convolutional Neural Networks
  C) Decision Trees
  D) Support Vector Machines



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: B
Explanation: Convolutional Neural Networks (CNNs) are a type of neural network specifically designed for image recognition tasks. They use convolutional and pooling layers to extract features from images, allowing them to learn complex patterns and classify objects with high accuracy. This makes them the most suitable choice for image recognition tasks.
Review in video at: 0:04 - 2:05

Question 2: What is the primary function of the activation function in a neural network neuron?
  A) To reduce the dimensionality of the input data
  B) To introduce non-linearity into the neuron's output
  C) To increase the number of layers in the network
  D) To decrease the number of neurons in a layer



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: B
Explanation: The activation function is used to introduce non-linearity into the neuron's output, allowing the neural network to learn and represent more complex relationships between inputs and outputs. Without an activation function, the neuron's output would be a linear combination of its inputs, limiting the network's ability to learn and generalize.
Review in video at: 2:05 - 4:06

Question 3: What is the primary function of the hidden layers in a neural network used for digit recognition and pattern analysis?
  A) To accept user input and display the output
  B) To apply non-linear transformations to the input data, allowing the network to learn complex patterns
  C) To store the network's weights and biases
  D) To perform the final classification or prediction


...

Incorrect!
Correct Answer: B
Explanation: Matrix-vector multiplication is used to transform input data into a higher-dimensional space, which is a fundamental operation in neural networks. This transformation allows the model to learn complex relationships between the input data and the output. In linear algebra, matrix-vector multiplication is used to apply linear transformations to vectors, which is essential for neural network functionality.
Review in video at: 14:14 - 16:15

Question 9: Which of the following activation functions is commonly used in the hidden layers of a neural network due to its ability to introduce non-linearity without saturating like the sigmoid function?
  A) Sigmoid function
  B) ReLU function
  C) Tanh function
  D) Linear function



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: B
Explanation: The ReLU (Rectified Linear Unit) function is commonly used in the hidden layers of a neural network because it introduces non-linearity without saturating like the sigmoid function. ReLU maps all negative values to 0 and all positive values to the same value, which helps to avoid the vanishing gradient problem that can occur with sigmoid. The sigmoid function, on the other hand, can saturate, meaning that its output will be close to 0 or 1 for large input values, which can lead to vanishing gradients during backpropagation. The tanh function also saturates, although it has a range of -1 to 1 instead of 0 to 1 like sigmoid. The linear function does not introduce non-linearity, so it is not typically used as an activation function in hidden layers.
Review in video at: 16:15 - 18:15

Question 10: What is the primary purpose of the ReLU activation function in Deep Neural Networks?
  A) To introduce non-linearity and allow the network to learn compl


Your answer (A/B/C/D):  a


Correct!
Correct Answer: A
Explanation: ReLU, or Rectified Linear Unit, is a widely used activation function in Deep Neural Networks. Its primary purpose is to introduce non-linearity into the network, allowing it to learn complex relationships between inputs and outputs. However, one of the drawbacks of ReLU is that it can result in 'dying neurons', where neurons with negative inputs become stuck in a state where they do not contribute to the network's output. This is because ReLU outputs 0 for any negative input, effectively 'killing' the neuron.
Review in video at: 18:15 - 18:25

Final Score: 1/10

Weak Topics: ['Machine Learning', 'Weights and Biases', 'Sigmoid Function', 'Sigmoid and ReLU Functions', 'Neural Network Structure', 'Neural Network Training', 'Neural Network Layers', 'Linear Algebra', 'Activation Functions', 'Neural Networks', 'Neural Network Functionality', 'Neural Network Architecture', 'Activation and Information Processing', 'Network Learning', 'Feature Extraction'

In [32]:
def run_pipeline(url, num_questions=10):
    # Agent 1
    agent1 = TranscriptionAgent()
    chunks = agent1.run(url)
    if not chunks:
        return None
    print(f"Agent 1 done: {len(chunks)} chunks")

    # Agent 1.5: Filter
    agent_filter = FilterAgent(llm)
    filtered_chunks = agent_filter.run(chunks)
    if not filtered_chunks:
        return None
    
    # Select chunks from filtered ones
    total_chunks = len(filtered_chunks)
    if total_chunks <= num_questions:
        selected_chunks = filtered_chunks
    else:
        step = total_chunks // num_questions
        selected_chunks = [filtered_chunks[i * step] for i in range(num_questions)]
    print(f"Selected {len(selected_chunks)} chunks for quiz")
    
    # Agent 2: Extract concepts
    agent2 = ConceptExtractionAgent(llm)
    concepts = agent2.run(selected_chunks)
    if not concepts:
        return None
    else:
        print(f"Agent 2 done: {len(concepts)} concept sets")
     
    # Agent 3
    agent3 = QuestionGenerationAgent(llm)
    quiz = agent3.run(concepts)
    if not quiz:
        return None
    else:
        print(f"Agent 3 done: {len(quiz['questions'])} questions")
    
    # Agent 4
    agent4 = EvaluationAgent(llm)
    results = agent4.run(quiz)
    
    # Return results
    return results

In [33]:
results = run_pipeline("https://www.youtube.com/watch?v=_gGRBWaRpAQ&list=PLZoTAELRMXVNNrHSKv36Lr3_156yCo6Nn&index=5")

Agent 1 done: 37 chunks
Skipping filler chunk at 14s
Skipping filler chunk at 134s
Skipping filler chunk at 254s
Skipping filler chunk at 3295s
Skipping filler chunk at 3415s
Skipping filler chunk at 3537s
Skipping filler chunk at 3780s
Skipping filler chunk at 4146s
Skipping filler chunk at 4267s
Skipping filler chunk at 4391s
Kept 27 out of 37 chunks
Selected 10 chunks for quiz
Skipping chunk: Expecting ',' delimiter: line 1 column 103 (char 102)
Agent 2 done: 9 concept sets
Agent 3 done: 9 questions

Question 1: Which of the following is a common step in text preprocessing for a Bag of Words model that helps reduce noise in the data by removing frequently occurring words like 'the' and 'and'?
  A) Tokenization
  B) Stop word removal
  C) Stemming
  D) Named entity recognition



Your answer (A/B/C/D):  c


Incorrect!
Correct Answer: B
Explanation: Stop word removal is the process of eliminating common words like 'the', 'and', 'a', etc. that do not add much value to the meaning of the text. These words are usually very frequent and can overshadow the importance of other words in the text, thereby reducing the model's performance. By removing stop words, we can reduce noise in the data and improve the model's ability to capture meaningful patterns.
Review in video at: 6:21 - 8:23

Question 2: In natural language processing, when using vector representation to represent words in a high-dimensional space, how can out-of-vocabulary words be handled in a sparse matrix representation of a text document?
  A) By assigning a fixed vector to all out-of-vocabulary words
  B) By using a subword modeling technique or a hash table to dynamically assign vectors to unseen words
  C) By removing the out-of-vocabulary words from the document
  D) By using a dense matrix representation instead



Your answer (A/B/C/D):  c


Incorrect!
Correct Answer: B
Explanation: This is correct because out-of-vocabulary words can be handled using subword modeling techniques, such as WordPiece or character-level embeddings, which can generate vector representations for unseen words by breaking them down into subwords or characters. Alternatively, hash tables can be used to dynamically assign vectors to out-of-vocabulary words. This approach allows the model to handle unseen words without having to assign a fixed vector to all out-of-vocabulary words, removing them from the document, or using a dense matrix representation.
Review in video at: 10:24 - 12:26

Question 3: What is the primary purpose of using TF-IDF in conjunction with Cosine Similarity to capture semantic meaning in text analysis?
  A) To increase the weight of rare words and decrease the weight of common words, then compare documents based on keyword matching
  B) To decrease the weight of common words and increase the weight of rare words, then compare do

...

Incorrect!
Correct Answer: B
Explanation: The correct answer is B because the term frequency-importance, or TF-IDF, is calculated as the product of the term frequency (TF) and the inverse document frequency (IDF). Here, TF is given as 5 (the term frequency of the word 'machine'), and IDF is given as log2(1000/5), which represents the logarithmic calculation of the total number of documents divided by the number of documents containing the word 'machine'. The vector multiplication given in the question is not necessary for this calculation, making option B the correct choice.
Review in video at: 30:37 - 32:38

Question 8: Which of the following text preprocessing steps reduces words to their base or root form, but does not always result in a valid word?
  A) Stopword removal
  B) Stemming
  C) Lemmatization
  D) Tokenization



Your answer (A/B/C/D):  b


Correct!
Correct Answer: B
Explanation: Stemming reduces words to their base or root form, but does not always result in a valid word. For example, the word 'running' might be stemmed to 'run', but the word 'happiness' might be stemmed to 'happi', which is not a valid word. Lemmatization, on the other hand, reduces words to their base or root form, known as the lemma, which is always a valid word. Stopword removal removes common words like 'the' and 'and', and tokenization splits text into individual words or tokens.
Review in video at: 38:44 - 40:44

Question 9: In a text processing task using N-grams, which of the following is a consequence of the sparsity problem?
  A) Increased dimensionality of the feature space leads to faster training times
  B) Large portions of the feature space are empty, making it difficult to train accurate models
  C) N-grams are unable to capture long-range dependencies in text
  D) The sparsity problem only occurs when using unigrams



Your answer (A/B/C/D):  c


Incorrect!
Correct Answer: B
Explanation: The sparsity problem in N-grams refers to the fact that as the size of the N-grams increases, the number of possible unique N-grams grows exponentially, resulting in a high-dimensional feature space where most features are zero, making it challenging to train accurate models. This is because large portions of the feature space are empty, as most N-grams do not appear in the training data.
Review in video at: 42:47 - 44:48

Final Score: 4/9

Weak Topics: ['Bag of Words', 'Stop Words', 'N-grams', 'Text Processing', 'Sparsity Problem', 'Text Preprocessing', 'Sparse Matrix', 'Vector Multiplication', 'Vector Representation', 'Inverse Document Frequency (IDF)', 'Logarithmic Calculation', 'Term Frequency', 'Out-of-Vocabulary Words', 'Term Frequency Importance']
